# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itz-me-sree/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir())

Current folder:
/content/flyrank-ml-internship

Files/folders here:
['.github', 'README.md', 'AGENTS.md', '.git', 'DATA_USE.md', 'submission', 'docs', '.gitignore', 'work', 'GUIDE.md', 'requirements.txt', 'LICENSE', 'outputs', 'notebooks', 'skills', 'scripts', 'data', 'SETUP.md', 'CLAUDE.md']


In [10]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumn names:")
print(df.columns.tolist())

Rows: 30000
Columns: 44

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [11]:
!git clone https://github.com/itz-me-sree/flyrank-ml-internship.git
%cd /content/flyrank-ml-internship

import os
print("Current folder:", os.getcwd())
print("Data exists:", os.path.exists("data/raw/content_refresh_anonymized.csv"))

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 139 (delta 51), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 1.86 MiB | 5.47 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/flyrank-ml-internship
Current folder: /content/flyrank-ml-internship
Data exists: True


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [12]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Features available before making a refresh recommendation
feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# Build feature matrix
X = df[feature_cols].copy()

# Replace infinite values
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing numeric values with median
X = X.fillna(X.median(numeric_only=True))

print("Feature matrix shape:", X.shape)
print("\nFeatures used:")
print(X.columns.tolist())

print("\nMissing values after filling:")
print(X.isna().sum().sum())

Feature matrix shape: (30000, 27)

Features used:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Missing values after filling:
0


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Notes

The feature vector uses measurable content, search-demand, historical-performance,
engagement, and freshness signals that can be observed before making a refresh
recommendation.

- `search_volume` — estimated search demand for the content topic.
- `competition` — competition level associated with the search opportunity.
- `cpc` — cost-per-click signal associated with the topic.
- `word_count` and `char_count` — measures of content size.
- `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, and `users_90d`
  — historical performance measures.
- `engaged_sessions_90d` and `scroll_events_90d` — engagement signals.
- `days_with_impressions` and `days_with_sessions` — activity coverage measures.
- `impressions_last_30d`, `clicks_last_30d`, and `sessions_last_30d` — recent
  performance.
- `impressions_prev_30d`, `clicks_prev_30d`, and `sessions_prev_30d` — previous
  period performance used as historical context.
- `content_age_days` and `days_since_last_update` — content freshness signals.
- `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, and `ai_traffic_pct`
  — derived performance and engagement signals.

Missing numeric values are replaced using the median of the available data.

Identifiers and fields that directly describe the observed performance trend or
encode derived categories are excluded from the model feature vector to reduce
the risk of leakage.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage Hunt

Potential leakage can occur when a feature contains information from the
outcome being predicted or directly encodes the observed performance trend.

The following fields are therefore excluded from the feature vector:

- `trend_direction`
- `trend_pct`

These fields describe the observed direction or magnitude of performance change
and could reveal the target.

Identifiers such as `content_id` and `client_id` are also excluded because they
do not represent generalizable content characteristics.

The model uses historical and content-level signals rather than future outcome
information.

In [15]:
# Check that known leakage-prone fields are not in the feature vector

leakage_candidates = [
    "trend_direction",
    "trend_pct"
]

found_leakage = [col for col in leakage_candidates if col in X.columns]

print("Leakage-prone fields found in feature vector:", found_leakage)

if not found_leakage:
    print("Leakage check passed.")
else:
    print("WARNING: Review these fields before modeling.")

Leakage-prone fields found in feature vector: []
Leakage check passed.


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded Fields and Reasons

- `content_id` — identifier; not a predictive content characteristic.
- `client_id` — identifier and privacy-sensitive; excluded from modeling.
- `provider_used` — implementation metadata, not required for the core task.
- `model_used` — implementation metadata, not required for the core task.
- `trend_direction` — directly describes performance direction and may leak the target.
- `trend_pct` — directly describes the magnitude of performance change and may leak the target.
- `age_tier`, `age_tier_order` — derived versions of content age; the numeric `content_age_days` feature is sufficient.
- `freshness_tier` — derived from freshness-related variables and not required when using numeric freshness measures.
- `word_count_tier` — derived from `word_count`.
- `char_count_tier` — derived from `char_count`.
- `impression_tier` — derived from impression values.
- `position_tier` — derived from average position.
- `content_type` and `main_intent` — categorical/context fields excluded from this first baseline feature vector.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.